# 01 - Dataset Familiarization

This notebook contains initial exploration and notes for the Amsterdam dataset.

In [48]:
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/amsterdam/2026-06-15")

expected_files = [
    "listings.csv.gz",
    "calendar.csv.gz",
    "reviews.csv.gz",
    "listings.csv",
    "reviews.csv",
    "neighbourhoods.csv",
    "neighbourhoods.geojson"
]

for file_name in expected_files:
    file_path = RAW_DATA_PATH / file_name
    print(file_name, "✅ Found" if file_path.exists() else "❌ Missing")

listings.csv.gz ✅ Found
calendar.csv.gz ✅ Found
reviews.csv.gz ✅ Found
listings.csv ✅ Found
reviews.csv ✅ Found
neighbourhoods.csv ✅ Found
neighbourhoods.geojson ✅ Found


In [49]:
import pandas as pd
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/amsterdam/2026-06-15")

files = {
    "detailed_listings": RAW_DATA_PATH / "listings.csv.gz",
    "calendar": RAW_DATA_PATH / "calendar.csv.gz",
    "detailed_reviews": RAW_DATA_PATH / "reviews.csv.gz",
    "summary_listings": RAW_DATA_PATH / "listings.csv",
    "summary_reviews": RAW_DATA_PATH / "reviews.csv",
    "neighbourhoods": RAW_DATA_PATH / "neighbourhoods.csv",
    "neighbourhoods_geojson": RAW_DATA_PATH / "neighbourhoods.geojson"
}

for name, path in files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'} - {path}")

detailed_listings: FOUND - ..\data\raw\amsterdam\2026-06-15\listings.csv.gz
calendar: FOUND - ..\data\raw\amsterdam\2026-06-15\calendar.csv.gz
detailed_reviews: FOUND - ..\data\raw\amsterdam\2026-06-15\reviews.csv.gz
summary_listings: FOUND - ..\data\raw\amsterdam\2026-06-15\listings.csv
summary_reviews: FOUND - ..\data\raw\amsterdam\2026-06-15\reviews.csv
neighbourhoods: FOUND - ..\data\raw\amsterdam\2026-06-15\neighbourhoods.csv
neighbourhoods_geojson: FOUND - ..\data\raw\amsterdam\2026-06-15\neighbourhoods.geojson


In [50]:
datasets = {}

datasets["detailed_listings"] = pd.read_csv(files["detailed_listings"])
datasets["calendar"] = pd.read_csv(files["calendar"])
datasets["detailed_reviews"] = pd.read_csv(files["detailed_reviews"])
datasets["summary_listings"] = pd.read_csv(files["summary_listings"])
datasets["summary_reviews"] = pd.read_csv(files["summary_reviews"])
datasets["neighbourhoods"] = pd.read_csv(files["neighbourhoods"])

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print(df.head(3))

detailed_listings
Rows: 10369
Columns: 90
      id                         listing_url       scrape_id last_scraped  \
0  28871  https://www.airbnb.com/rooms/28871  20260615212022   2026-06-16   
1  29051  https://www.airbnb.com/rooms/29051  20260615212022   2026-06-16   
2  44129  https://www.airbnb.com/rooms/44129  20260615212022   2026-06-24   

            source                              name  \
0  previous scrape           Comfortable double room   
1  previous scrape  Comfortable single / double room   
2  previous scrape     Luxury design with canal view   

                                         description  neighborhood_overview  \
0          Basic bedroom in the center of Amsterdam.                    NaN   
1  This room can also be rented as a single or a ...                    NaN   
2  Welcome to my little gem<br /><br />Cozy, brig...                    NaN   

                                         picture_url  host_id  ...  \
0  https://a0.muscache.com/pictures/1

## 1. Dataset Inventory

This section summarizes all raw Amsterdam datasets loaded from Inside Airbnb.  
The goal is to understand the size and structure of each file before cleaning or analysis.

In [51]:
import pandas as pd
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/amsterdam/2026-06-15")

files = {
    "detailed_listings": RAW_DATA_PATH / "listings.csv.gz",
    "calendar": RAW_DATA_PATH / "calendar.csv.gz",
    "detailed_reviews": RAW_DATA_PATH / "reviews.csv.gz",
    "summary_listings": RAW_DATA_PATH / "listings.csv",
    "summary_reviews": RAW_DATA_PATH / "reviews.csv",
    "neighbourhoods": RAW_DATA_PATH / "neighbourhoods.csv"
}

datasets = {
    "detailed_listings": pd.read_csv(files["detailed_listings"]),
    "calendar": pd.read_csv(files["calendar"]),
    "detailed_reviews": pd.read_csv(files["detailed_reviews"]),
    "summary_listings": pd.read_csv(files["summary_listings"]),
    "summary_reviews": pd.read_csv(files["summary_reviews"]),
    "neighbourhoods": pd.read_csv(files["neighbourhoods"])
}

inventory = []

for name, df in datasets.items():
    inventory.append({
        "dataset_name": name,
        "file_name": files[name].name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

inventory_df = pd.DataFrame(inventory)
inventory_df

,dataset_name,file_name,rows,columns,memory_mb
0,detailed_listings,listings.csv.gz,10369,90,28.00
1,calendar,calendar.csv.gz,3819725,5,185.78
2,detailed_reviews,reviews.csv.gz,545162,6,164.49
3,summary_listings,listings.csv,10465,19,2.57
4,summary_reviews,reviews.csv,545162,2,13.52
5,neighbourhoods,neighbourhoods.csv,22,2,0.00


In [52]:
REPORTS_PATH = Path("../reports/data_quality")
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

inventory_df.to_csv(REPORTS_PATH / "amsterdam_dataset_inventory.csv", index=False)

print("Dataset inventory saved successfully.")

Dataset inventory saved successfully.


### Initial Observation

The Amsterdam dataset contains multiple raw files at different levels of detail.  
The detailed listings file contains rich listing, host, pricing, location, and review score attributes.  
The calendar file provides daily availability and pricing records, while the reviews file provides review-level data.  
The neighbourhood file supports geographic grouping and filtering.

This confirms that the dataset can support data engineering, exploratory analysis, statistical testing, machine learning, and NLP-based review analysis.

## 2. Schema Profiling

This section creates a column-level schema profile for each raw dataset.  
For every column, the profile records data type, missing values, uniqueness, and sample values.  
This helps identify columns that require cleaning, type conversion, validation, or special interpretation.

In [53]:
def create_schema_profile(df, dataset_name):
    """
    Create a column-level schema profile for a dataframe.
    """
    profile_rows = []

    for column in df.columns:
        sample_values = (
            df[column]
            .dropna()
            .astype(str)
            .unique()[:5]
        )

        profile_rows.append({
            "dataset_name": dataset_name,
            "column_name": column,
            "data_type": str(df[column].dtype),
            "non_null_count": df[column].notna().sum(),
            "null_count": df[column].isna().sum(),
            "null_percentage": round((df[column].isna().sum() / len(df)) * 100, 2),
            "unique_count": df[column].nunique(dropna=True),
            "sample_values": " | ".join(sample_values)
        })

    return pd.DataFrame(profile_rows)

In [54]:
schema_profiles = []

for dataset_name, df in datasets.items():
    schema_profile = create_schema_profile(df, dataset_name)
    schema_profiles.append(schema_profile)

all_schema_profiles_df = pd.concat(schema_profiles, ignore_index=True)

all_schema_profiles_df.head(20)

,dataset_name,column_name,data_type,non_null_count,null_count,null_percentage,unique_count,sample_values
0,detailed_listings,id,int64,10369,0,0.00,10369,28871 | 29051 | 44129 | 44391 | 48373
1,detailed_listings,listing_url,str,10369,0,0.00,10369,https://www.airbnb.com/rooms/28871 | https://w...
2,detailed_listings,scrape_id,int64,10369,0,0.00,1,20260615212022
3,detailed_listings,last_scraped,str,10369,0,0.00,3,2026-06-16 | 2026-06-24 | 2026-06-15
4,detailed_listings,source,str,10369,0,0.00,2,previous scrape | city scrape
5,detailed_listings,name,str,10369,0,0.00,10076,Comfortable double room | Comfortable single /...
6,detailed_listings,description,str,9919,450,4.34,9640,Basic bedroom in the center of Amsterdam. | Th...
7,detailed_listings,neighborhood_overview,float64,0,10369,100.00,0,
8,detailed_listings,picture_url,str,10369,0,0.00,10315,https://a0.muscache.com/pictures/160889/362340...
9,detailed_listings,host_id,int64,10369,0,0.00,9104,124245 | 187728 | 194779 | 220434 | 225987


In [55]:
schema_output_path = REPORTS_PATH / "amsterdam_schema_profile.csv"

all_schema_profiles_df.to_csv(schema_output_path, index=False)

print(f"Schema profile saved to: {schema_output_path}")

PermissionError: [Errno 13] Permission denied: '..\\reports\\data_quality\\amsterdam_schema_profile.csv'

## 3. Missing Values and Data Quality Analysis

This section evaluates the completeness and basic quality of each raw dataset.  
The goal is to identify missing values, duplicate records, key column issues, and domain validation problems before cleaning and transformation.

These checks help decide which columns can be trusted, which columns need cleaning, and which columns should be excluded from downstream analysis.

In [56]:
missing_value_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        null_count = df[column].isna().sum()
        null_percentage = round((null_count / len(df)) * 100, 2)

        missing_value_rows.append({
            "dataset_name": dataset_name,
            "column_name": column,
            "total_rows": len(df),
            "null_count": null_count,
            "null_percentage": null_percentage
        })

missing_values_df = pd.DataFrame(missing_value_rows)

missing_values_df = missing_values_df.sort_values(
    by=["dataset_name", "null_percentage"],
    ascending=[True, False]
)

missing_values_df.head(30)

,dataset_name,column_name,total_rows,null_count,null_percentage
90,calendar,listing_id,3819725,0,0.00
91,calendar,date,3819725,0,0.00
92,calendar,available,3819725,0,0.00
93,calendar,minimum_nights,3819725,0,0.00
94,calendar,maximum_nights,3819725,0,0.00
7,detailed_listings,neighborhood_overview,10369,10369,100.00
14,detailed_listings,host_since,10369,10369,100.00
21,detailed_listings,host_response_time,10369,10369,100.00
22,detailed_listings,host_response_rate,10369,10369,100.00
23,detailed_listings,host_acceptance_rate,10369,10369,100.00


In [57]:
missing_output_path = REPORTS_PATH / "amsterdam_missing_values_summary.csv"

missing_values_df.to_csv(missing_output_path, index=False)

print(f"Missing values summary saved to: {missing_output_path}")

Missing values summary saved to: ..\reports\data_quality\amsterdam_missing_values_summary.csv


### Duplicate Record Check

This check identifies exact duplicate rows in each raw dataset.  
Exact duplicates can indicate repeated ingestion, scraping duplication, or source-level data quality issues.

In [58]:
duplicate_summary_rows = []

for dataset_name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = round((duplicate_count / len(df)) * 100, 4)

    duplicate_summary_rows.append({
        "dataset_name": dataset_name,
        "total_rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": duplicate_percentage
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary_rows)
duplicate_summary_df

,dataset_name,total_rows,duplicate_rows,duplicate_percentage
0,detailed_listings,10369,0,0.0000
1,calendar,3819725,0,0.0000
2,detailed_reviews,545162,0,0.0000
3,summary_listings,10465,0,0.0000
4,summary_reviews,545162,22541,4.1347
5,neighbourhoods,22,0,0.0000


In [59]:
duplicate_output_path = REPORTS_PATH / "amsterdam_duplicate_summary.csv"

duplicate_summary_df.to_csv(duplicate_output_path, index=False)

print(f"Duplicate summary saved to: {duplicate_output_path}")

Duplicate summary saved to: ..\reports\data_quality\amsterdam_duplicate_summary.csv


### Key Column Quality Check

This check evaluates whether important ID columns are complete and unique.  
These columns are important because they are used to connect listings, calendar records, reviews, hosts, and neighbourhoods.

In [60]:
key_checks = [
    {
        "dataset_name": "detailed_listings",
        "key_column": "id",
        "expected_role": "Primary key for detailed listings"
    },
    {
        "dataset_name": "summary_listings",
        "key_column": "id",
        "expected_role": "Primary key for summary listings"
    },
    {
        "dataset_name": "calendar",
        "key_column": "listing_id",
        "expected_role": "Foreign key referencing listings"
    },
    {
        "dataset_name": "detailed_reviews",
        "key_column": "listing_id",
        "expected_role": "Foreign key referencing listings"
    },
    {
        "dataset_name": "summary_reviews",
        "key_column": "listing_id",
        "expected_role": "Foreign key referencing listings"
    },
    {
        "dataset_name": "detailed_listings",
        "key_column": "host_id",
        "expected_role": "Host identifier"
    }
]

key_quality_rows = []

for check in key_checks:
    dataset_name = check["dataset_name"]
    key_column = check["key_column"]
    df = datasets[dataset_name]

    if key_column in df.columns:
        key_quality_rows.append({
            "dataset_name": dataset_name,
            "key_column": key_column,
            "expected_role": check["expected_role"],
            "total_rows": len(df),
            "non_null_count": df[key_column].notna().sum(),
            "null_count": df[key_column].isna().sum(),
            "unique_count": df[key_column].nunique(dropna=True),
            "duplicate_key_count": df[key_column].duplicated().sum()
        })
    else:
        key_quality_rows.append({
            "dataset_name": dataset_name,
            "key_column": key_column,
            "expected_role": check["expected_role"],
            "total_rows": len(df),
            "non_null_count": None,
            "null_count": None,
            "unique_count": None,
            "duplicate_key_count": None
        })

key_quality_df = pd.DataFrame(key_quality_rows)
key_quality_df

,dataset_name,key_column,expected_role,total_rows,non_null_count,null_count,unique_count,duplicate_key_count
0,detailed_listings,id,Primary key for detailed listings,10369,10369,0,10369,0
1,summary_listings,id,Primary key for summary listings,10465,10465,0,10465,0
2,calendar,listing_id,Foreign key referencing listings,3819725,3819725,0,10465,3809260
3,detailed_reviews,listing_id,Foreign key referencing listings,545162,545162,0,9432,535730
4,summary_reviews,listing_id,Foreign key referencing listings,545162,545162,0,9432,535730
5,detailed_listings,host_id,Host identifier,10369,10369,0,9104,1265


In [61]:
key_quality_output_path = REPORTS_PATH / "amsterdam_key_quality_summary.csv"

key_quality_df.to_csv(key_quality_output_path, index=False)

print(f"Key quality summary saved to: {key_quality_output_path}")

Key quality summary saved to: ..\reports\data_quality\amsterdam_key_quality_summary.csv


### Domain Validation Checks

This section validates important fields against basic business rules.

Examples:
- Price should not be negative.
- Latitude should be between -90 and 90.
- Longitude should be between -180 and 180.
- Availability values should be valid.
- Review scores should fall within expected rating ranges.

In [62]:
def clean_price_value(value):
    """
    Convert price values like '$1,234.00' into numeric values.
    Returns NaN if the value cannot be converted.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value)
    value = value.replace("$", "").replace(",", "").strip()

    try:
        return float(value)
    except ValueError:
        return pd.NA


validation_rows = []

# Detailed listings price validation
if "price" in datasets["detailed_listings"].columns:
    detailed_price = datasets["detailed_listings"]["price"].apply(clean_price_value)
    validation_rows.append({
        "dataset_name": "detailed_listings",
        "field": "price",
        "rule": "Price should be numeric and non-negative",
        "invalid_count": ((detailed_price.notna()) & (detailed_price < 0)).sum(),
        "missing_or_unparseable_count": detailed_price.isna().sum()
    })

# Summary listings price validation
if "price" in datasets["summary_listings"].columns:
    summary_price = pd.to_numeric(datasets["summary_listings"]["price"], errors="coerce")
    validation_rows.append({
        "dataset_name": "summary_listings",
        "field": "price",
        "rule": "Price should be numeric and non-negative",
        "invalid_count": ((summary_price.notna()) & (summary_price < 0)).sum(),
        "missing_or_unparseable_count": summary_price.isna().sum()
    })

# Calendar price validation
if "price" in datasets["calendar"].columns:
    calendar_price = datasets["calendar"]["price"].apply(clean_price_value)
    validation_rows.append({
        "dataset_name": "calendar",
        "field": "price",
        "rule": "Daily price should be numeric and non-negative",
        "invalid_count": ((calendar_price.notna()) & (calendar_price < 0)).sum(),
        "missing_or_unparseable_count": calendar_price.isna().sum()
    })

# Latitude validation
for dataset_name in ["detailed_listings", "summary_listings"]:
    if "latitude" in datasets[dataset_name].columns:
        latitude = pd.to_numeric(datasets[dataset_name]["latitude"], errors="coerce")
        validation_rows.append({
            "dataset_name": dataset_name,
            "field": "latitude",
            "rule": "Latitude should be between -90 and 90",
            "invalid_count": ((latitude.notna()) & ((latitude < -90) | (latitude > 90))).sum(),
            "missing_or_unparseable_count": latitude.isna().sum()
        })

# Longitude validation
for dataset_name in ["detailed_listings", "summary_listings"]:
    if "longitude" in datasets[dataset_name].columns:
        longitude = pd.to_numeric(datasets[dataset_name]["longitude"], errors="coerce")
        validation_rows.append({
            "dataset_name": dataset_name,
            "field": "longitude",
            "rule": "Longitude should be between -180 and 180",
            "invalid_count": ((longitude.notna()) & ((longitude < -180) | (longitude > 180))).sum(),
            "missing_or_unparseable_count": longitude.isna().sum()
        })

# Calendar availability validation
if "available" in datasets["calendar"].columns:
    valid_available_values = {"t", "f", True, False}
    available_values = datasets["calendar"]["available"]
    validation_rows.append({
        "dataset_name": "calendar",
        "field": "available",
        "rule": "Availability should be t/f or boolean",
        "invalid_count": (~available_values.isin(valid_available_values)).sum(),
        "missing_or_unparseable_count": available_values.isna().sum()
    })

# Review score validation
review_score_columns = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for column in review_score_columns:
    if column in datasets["detailed_listings"].columns:
        score = pd.to_numeric(datasets["detailed_listings"][column], errors="coerce")
        validation_rows.append({
            "dataset_name": "detailed_listings",
            "field": column,
            "rule": "Review score should be between 0 and 5",
            "invalid_count": ((score.notna()) & ((score < 0) | (score > 5))).sum(),
            "missing_or_unparseable_count": score.isna().sum()
        })

domain_validation_df = pd.DataFrame(validation_rows)
domain_validation_df

,dataset_name,field,rule,invalid_count,missing_or_unparseable_count
0,detailed_listings,price,Price should be numeric and non-negative,0,3992
1,summary_listings,price,Price should be numeric and non-negative,0,3994
2,detailed_listings,latitude,Latitude should be between -90 and 90,0,0
3,summary_listings,latitude,Latitude should be between -90 and 90,0,0
4,detailed_listings,longitude,Longitude should be between -180 and 180,0,0
5,summary_listings,longitude,Longitude should be between -180 and 180,0,0
6,calendar,available,Availability should be t/f or boolean,0,0
7,detailed_listings,review_scores_rating,Review score should be between 0 and 5,0,1023
8,detailed_listings,review_scores_accuracy,Review score should be between 0 and 5,0,1023
9,detailed_listings,review_scores_cleanliness,Review score should be between 0 and 5,0,1024


In [63]:
domain_validation_output_path = REPORTS_PATH / "amsterdam_domain_validation_summary.csv"

domain_validation_df.to_csv(domain_validation_output_path, index=False)

print(f"Domain validation summary saved to: {domain_validation_output_path}")

Domain validation summary saved to: ..\reports\data_quality\amsterdam_domain_validation_summary.csv


### Data Quality Observations

Initial data quality profiling shows that the Amsterdam dataset is large enough for meaningful engineering, analytics, and machine learning work.  
The calendar dataset is the largest table because it stores daily availability and price records for each listing.  
The reviews dataset is also large and can support time-based demand analysis and NLP experiments.

Some columns contain high missing-value percentages, which indicates that not all raw fields are suitable for downstream analysis.  
The next stage will focus on selecting useful columns, cleaning price/date fields, validating key relationships, and preparing analytics-ready datasets.

## 4. Relationship Mapping Between Files

This section maps the relationships between the raw Inside Airbnb files.  
The purpose is to identify primary keys, foreign keys, row-level grain, and join paths between listings, calendar records, reviews, hosts, and neighbourhoods.

Understanding these relationships is required before creating cleaned datasets, enriched master tables, and dimensional models.

In [66]:
relationship_rows = [
    {
        "source_dataset": "detailed_listings",
        "source_column": "id",
        "target_dataset": "calendar",
        "target_column": "listing_id",
        "relationship_type": "One listing to many calendar rows",
        "business_meaning": "Each listing can have daily availability and price records."
    },
    {
        "source_dataset": "detailed_listings",
        "source_column": "id",
        "target_dataset": "detailed_reviews",
        "target_column": "listing_id",
        "relationship_type": "One listing to many review rows",
        "business_meaning": "Each listing can receive multiple guest reviews."
    },
    {
        "source_dataset": "detailed_listings",
        "source_column": "id",
        "target_dataset": "summary_reviews",
        "target_column": "listing_id",
        "relationship_type": "One listing to many review-date rows",
        "business_meaning": "Each listing can have multiple review dates for time-based demand analysis."
    },
    {
        "source_dataset": "detailed_listings",
        "source_column": "host_id",
        "target_dataset": "derived_host_dimension",
        "target_column": "host_id",
        "relationship_type": "Many listings to one host",
        "business_meaning": "One host may manage one or many listings."
    },
    {
        "source_dataset": "detailed_listings",
        "source_column": "neighbourhood_cleansed",
        "target_dataset": "neighbourhoods",
        "target_column": "neighbourhood",
        "relationship_type": "Many listings to one neighbourhood",
        "business_meaning": "Each listing belongs to one neighbourhood group used for geographic analysis."
    }
]

relationships_df = pd.DataFrame(relationship_rows)
relationships_df

,source_dataset,source_column,target_dataset,target_column,relationship_type,business_meaning
0,detailed_listings,id,calendar,listing_id,One listing to many calendar rows,Each listing can have daily availability and p...
1,detailed_listings,id,detailed_reviews,listing_id,One listing to many review rows,Each listing can receive multiple guest reviews.
2,detailed_listings,id,summary_reviews,listing_id,One listing to many review-date rows,Each listing can have multiple review dates fo...
3,detailed_listings,host_id,derived_host_dimension,host_id,Many listings to one host,One host may manage one or many listings.
4,detailed_listings,neighbourhood_cleansed,neighbourhoods,neighbourhood,Many listings to one neighbourhood,Each listing belongs to one neighbourhood grou...


In [67]:
relationship_output_path = REPORTS_PATH / "amsterdam_relationship_mapping.csv"

relationships_df.to_csv(relationship_output_path, index=False)

print(f"Relationship mapping saved to: {relationship_output_path}")

Relationship mapping saved to: ..\reports\data_quality\amsterdam_relationship_mapping.csv


### Listing ID Coverage Check

This check compares listing identifiers across listings, calendar, and review datasets.  
It helps identify whether child records exist without a matching listing record in the main listings table.

In [68]:
detailed_listing_ids = set(datasets["detailed_listings"]["id"])
summary_listing_ids = set(datasets["summary_listings"]["id"])
calendar_listing_ids = set(datasets["calendar"]["listing_id"])
detailed_review_listing_ids = set(datasets["detailed_reviews"]["listing_id"])
summary_review_listing_ids = set(datasets["summary_reviews"]["listing_id"])

coverage_rows = []

coverage_checks = [
    {
        "reference_table": "detailed_listings",
        "reference_key": "id",
        "child_table": "summary_listings",
        "child_key": "id",
        "reference_ids": detailed_listing_ids,
        "child_ids": summary_listing_ids
    },
    {
        "reference_table": "detailed_listings",
        "reference_key": "id",
        "child_table": "calendar",
        "child_key": "listing_id",
        "reference_ids": detailed_listing_ids,
        "child_ids": calendar_listing_ids
    },
    {
        "reference_table": "detailed_listings",
        "reference_key": "id",
        "child_table": "detailed_reviews",
        "child_key": "listing_id",
        "reference_ids": detailed_listing_ids,
        "child_ids": detailed_review_listing_ids
    },
    {
        "reference_table": "detailed_listings",
        "reference_key": "id",
        "child_table": "summary_reviews",
        "child_key": "listing_id",
        "reference_ids": detailed_listing_ids,
        "child_ids": summary_review_listing_ids
    }
]

for check in coverage_checks:
    missing_from_reference = check["child_ids"] - check["reference_ids"]
    missing_from_child = check["reference_ids"] - check["child_ids"]

    coverage_rows.append({
        "reference_table": check["reference_table"],
        "reference_key": check["reference_key"],
        "child_table": check["child_table"],
        "child_key": check["child_key"],
        "reference_unique_ids": len(check["reference_ids"]),
        "child_unique_ids": len(check["child_ids"]),
        "child_ids_missing_in_reference": len(missing_from_reference),
        "reference_ids_missing_in_child": len(missing_from_child),
        "coverage_percentage": round(
            ((len(check["child_ids"]) - len(missing_from_reference)) / len(check["child_ids"])) * 100, 
            2
        )
    })

id_coverage_df = pd.DataFrame(coverage_rows)
id_coverage_df

,reference_table,reference_key,child_table,child_key,reference_unique_ids,child_unique_ids,child_ids_missing_in_reference,reference_ids_missing_in_child,coverage_percentage
0,detailed_listings,id,summary_listings,id,10369,10465,96,0,99.08
1,detailed_listings,id,calendar,listing_id,10369,10465,96,0,99.08
2,detailed_listings,id,detailed_reviews,listing_id,10369,9432,86,1023,99.09
3,detailed_listings,id,summary_reviews,listing_id,10369,9432,86,1023,99.09


In [69]:
id_coverage_output_path = REPORTS_PATH / "amsterdam_listing_id_coverage.csv"

id_coverage_df.to_csv(id_coverage_output_path, index=False)

print(f"Listing ID coverage report saved to: {id_coverage_output_path}")

Listing ID coverage report saved to: ..\reports\data_quality\amsterdam_listing_id_coverage.csv


### Relationship Mapping Observations

The listing identifier is the central join key across the Amsterdam Airbnb dataset.  
The detailed listings table provides the main listing-level attributes, while calendar and review files provide one-to-many records linked through listing identifiers.

The calendar table has many rows per listing because each listing can have daily availability and price records.  
The reviews table has many rows per listing because a listing can receive multiple guest reviews.

The host relationship is derived from the listings table using host_id.  
A single host may manage multiple listings, which supports host portfolio and supply concentration analysis.

Any listing ID coverage gaps between files will be treated as data lineage and completeness considerations during cleaning and modeling.

### Listing ID Coverage Observations

The summary listings table contains 10,465 unique listing IDs, while the detailed listings table contains 10,369 unique listing IDs.  
There are 96 listing IDs present in the summary listings and calendar files that are not present in the detailed listings file.

This indicates that the summary listings file provides the broadest listing coverage for the Amsterdam snapshot, while the detailed listings file provides richer descriptive and host-level attributes.

For downstream modeling, the project will use `summary_listings` as the base listing universe and enrich it with selected fields from `detailed_listings` where available.  
This approach preserves maximum listing coverage while still allowing richer analysis for listings with detailed metadata.

Listings without detailed metadata will be retained where possible and flagged during transformation, rather than removed without justification.

## 5. Cleaned Base Listing Table

This section creates a cleaned base listing table using the summary listings dataset.  
The summary listings table is selected as the base listing universe because it has the broadest listing coverage and aligns with the calendar dataset.

The cleaned table standardizes identifiers, prices, geographic fields, categorical values, and core numerical columns.  
This table will later be enriched with detailed listing, calendar, review, and neighbourhood-level features.

In [70]:
summary_listings = datasets["summary_listings"].copy()

print("Summary listings shape:", summary_listings.shape)
print("Columns:")
print(summary_listings.columns.tolist())

Summary listings shape: (10465, 19)
Columns:
['id', 'name', 'host_id', 'host_profile_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'license']


In [71]:
summary_listings.head()

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,28871,Comfortable double room,124245.0,1.462510e+18,Edwin,NaN,Centrum-West,52.367750,4.89092,Private room,94.0,1.0,799,2026-06-01,4.15,2.0,11,88,0363 607B EA74 0BD8 2F6F
1,29051,Comfortable single / double room,124245.0,1.462510e+18,Edwin,NaN,Centrum-Oost,52.365840,4.89111,Private room,NaN,1.0,906,2026-06-01,4.88,2.0,2,82,0363 607B EA74 0BD8 2F6F
2,44129,Luxury design with canal view,187728.0,1.462512e+18,Tanya,NaN,Centrum-West,52.382110,4.88630,Entire home/apt,314.0,3.0,186,2026-04-30,0.96,4.0,3,5,03635399E87602900F47
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779.0,1.462512e+18,Jan,NaN,Centrum-Oost,52.371680,4.91471,Entire home/apt,NaN,3.0,42,2022-08-20,0.22,1.0,0,0,0363 E76E F06A C1DD 172C
4,48373,Cozy family home in Amsterdam South,220434.0,1.462513e+18,Vesna & Misha,NaN,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,NaN,3.0,5,2024-04-28,0.14,1.0,0,0,0363 4A2B A6AD 0196 F684


### Cleaning Helper Functions

The following helper functions standardize common raw data issues such as text formatting, numeric conversion, date parsing, and price cleaning.

In [72]:
def standardize_text(value):
    """
    Standardize text values by trimming spaces.
    Returns NA when the value is missing.
    """
    if pd.isna(value):
        return pd.NA
    return str(value).strip()


def clean_numeric(value):
    """
    Convert a value to numeric.
    Invalid values are converted to NaN.
    """
    return pd.to_numeric(value, errors="coerce")


def parse_date(value):
    """
    Convert a value to datetime.
    Invalid dates are converted to NaT.
    """
    return pd.to_datetime(value, errors="coerce")


def clean_price(value):
    """
    Convert price values into numeric format.
    Works for values like '$1,200.00' and also plain numeric values.
    """
    if pd.isna(value):
        return pd.NA
    
    value = str(value).replace("$", "").replace(",", "").strip()
    return pd.to_numeric(value, errors="coerce")

In [73]:
summary_listings_clean = pd.DataFrame()

summary_listings_clean["listing_id"] = summary_listings["id"].astype("Int64")
summary_listings_clean["listing_name"] = summary_listings["name"].apply(standardize_text)

summary_listings_clean["host_id"] = summary_listings["host_id"].astype("Int64")
summary_listings_clean["host_profile_id"] = summary_listings["host_profile_id"].astype("Int64")
summary_listings_clean["host_name"] = summary_listings["host_name"].apply(standardize_text)

summary_listings_clean["neighbourhood_group"] = summary_listings["neighbourhood_group"].apply(standardize_text)
summary_listings_clean["neighbourhood"] = summary_listings["neighbourhood"].apply(standardize_text)

summary_listings_clean["latitude"] = pd.to_numeric(summary_listings["latitude"], errors="coerce").round(6)
summary_listings_clean["longitude"] = pd.to_numeric(summary_listings["longitude"], errors="coerce").round(6)

summary_listings_clean["room_type"] = summary_listings["room_type"].apply(standardize_text)
summary_listings_clean["price"] = summary_listings["price"].apply(clean_price)

summary_listings_clean["minimum_nights"] = pd.to_numeric(summary_listings["minimum_nights"], errors="coerce")
summary_listings_clean["number_of_reviews"] = pd.to_numeric(summary_listings["number_of_reviews"], errors="coerce")
summary_listings_clean["last_review"] = pd.to_datetime(summary_listings["last_review"], errors="coerce")
summary_listings_clean["reviews_per_month"] = pd.to_numeric(summary_listings["reviews_per_month"], errors="coerce")
summary_listings_clean["calculated_host_listings_count"] = pd.to_numeric(
    summary_listings["calculated_host_listings_count"], errors="coerce"
)
summary_listings_clean["availability_365"] = pd.to_numeric(summary_listings["availability_365"], errors="coerce")
summary_listings_clean["number_of_reviews_ltm"] = pd.to_numeric(summary_listings["number_of_reviews_ltm"], errors="coerce")
summary_listings_clean["license"] = summary_listings["license"].apply(standardize_text)

summary_listings_clean["city"] = "Amsterdam"
summary_listings_clean["country"] = "Netherlands"
summary_listings_clean["snapshot_date"] = pd.to_datetime("2026-06-15")

summary_listings_clean.head()

,listing_id,listing_name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,city,country,snapshot_date
0,28871,Comfortable double room,124245,1462510208428038912,Edwin,<NA>,Centrum-West,52.367750,4.89092,Private room,...,799,2026-06-01,4.15,2.0,11,88,0363 607B EA74 0BD8 2F6F,Amsterdam,Netherlands,2026-06-15
1,29051,Comfortable single / double room,124245,1462510208428038912,Edwin,<NA>,Centrum-Oost,52.365840,4.89111,Private room,...,906,2026-06-01,4.88,2.0,2,82,0363 607B EA74 0BD8 2F6F,Amsterdam,Netherlands,2026-06-15
2,44129,Luxury design with canal view,187728,1462512125293144576,Tanya,<NA>,Centrum-West,52.382110,4.88630,Entire home/apt,...,186,2026-04-30,0.96,4.0,3,5,03635399E87602900F47,Amsterdam,Netherlands,2026-06-15
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,1462512336903650304,Jan,<NA>,Centrum-Oost,52.371680,4.91471,Entire home/apt,...,42,2022-08-20,0.22,1.0,0,0,0363 E76E F06A C1DD 172C,Amsterdam,Netherlands,2026-06-15
4,48373,Cozy family home in Amsterdam South,220434,1462513002935125760,Vesna & Misha,<NA>,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,...,5,2024-04-28,0.14,1.0,0,0,0363 4A2B A6AD 0196 F684,Amsterdam,Netherlands,2026-06-15


In [78]:
clean_listing_validation = {
    "row_count": len(summary_listings_clean),
    "column_count": summary_listings_clean.shape[1],
    "unique_listing_ids": summary_listings_clean["listing_id"].nunique(),
    "duplicate_listing_ids": summary_listings_clean["listing_id"].duplicated().sum(),
    "missing_listing_ids": summary_listings_clean["listing_id"].isna().sum(),
    "missing_price_count": summary_listings_clean["price"].isna().sum(),
    "negative_price_count": (summary_listings_clean["price"] < 0).sum(),
    "invalid_latitude_count": (
        (summary_listings_clean["latitude"] < -90) | 
        (summary_listings_clean["latitude"] > 90)
    ).sum(),
    "invalid_longitude_count": (
        (summary_listings_clean["longitude"] < -180) | 
        (summary_listings_clean["longitude"] > 180)
    ).sum()
}

clean_listing_validation_df = pd.DataFrame(
    list(clean_listing_validation.items()),
    columns=["check_name", "check_value"]
)

clean_listing_validation_df

,check_name,check_value
0,row_count,10465
1,column_count,22
2,unique_listing_ids,10465
3,duplicate_listing_ids,0
4,missing_listing_ids,0
5,missing_price_count,3994
6,negative_price_count,0
7,invalid_latitude_count,0
8,invalid_longitude_count,0


In [79]:
PROCESSED_DATA_PATH = Path("../data/processed/amsterdam")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

summary_listings_clean_output_path = PROCESSED_DATA_PATH / "summary_listings_clean.parquet"

summary_listings_clean.to_parquet(summary_listings_clean_output_path, index=False)

print(f"Cleaned summary listings saved to: {summary_listings_clean_output_path}")

Cleaned summary listings saved to: ..\data\processed\amsterdam\summary_listings_clean.parquet


In [80]:
clean_listing_validation_output_path = REPORTS_PATH / "amsterdam_clean_listing_validation.csv"

clean_listing_validation_df.to_csv(clean_listing_validation_output_path, index=False)

print(f"Cleaned listing validation saved to: {clean_listing_validation_output_path}")

Cleaned listing validation saved to: ..\reports\data_quality\amsterdam_clean_listing_validation.csv


## 6. Detailed Listings Enrichment

This section enriches the cleaned base listing table with selected attributes from the detailed listings dataset.

The summary listings table is used as the base listing universe because it has the broadest coverage.  
The detailed listings table is used as an enrichment source because it contains richer host, property, amenity, and review score attributes.

Listings that do not have matching detailed metadata will be retained with null enriched fields.

In [82]:
detailed_listings = datasets["detailed_listings"].copy()

print("Detailed listings shape:", detailed_listings.shape)
print("Columns:")
print(detailed_listings.columns.tolist())

Detailed listings shape: (10369, 90)
Columns:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_pri

### Selected Enrichment Columns

Only selected detailed listing columns are used for enrichment.  
This avoids carrying unnecessary raw fields into the analytical layer and keeps the final listing master table focused and interpretable.

In [104]:
selected_detailed_columns = [
    "id",
    "host_since",
    "host_is_superhost",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_listings_count",
    "host_total_listings_count",
    "host_identity_verified",
    "property_type",
    "accommodates",
    "bathrooms",
    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities",
    "instant_bookable",
    "availability_30",
    "availability_60",
    "availability_90",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

available_detailed_columns = [
    column for column in selected_detailed_columns 
    if column in detailed_listings.columns
]

missing_detailed_columns = [
    column for column in selected_detailed_columns 
    if column not in detailed_listings.columns
]

print("Available columns:", available_detailed_columns)
print("Missing columns:", missing_detailed_columns)

Available columns: ['id', 'host_since', 'host_is_superhost', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_listings_count', 'host_total_listings_count', 'host_identity_verified', 'property_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'instant_bookable', 'availability_30', 'availability_60', 'availability_90', 'number_of_reviews_l30d', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value']
Missing columns: []


In [105]:
detailed_enrichment = detailed_listings[available_detailed_columns].copy()

detailed_enrichment = detailed_enrichment.rename(columns={
    "id": "listing_id"
})

detailed_enrichment.head()

,listing_id,host_since,host_is_superhost,host_response_time,host_response_rate,host_acceptance_rate,host_listings_count,host_total_listings_count,host_identity_verified,property_type,...,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
0,28871,NaN,t,NaN,NaN,NaN,2.0,NaN,t,Private room in rental unit,...,95,255,23970.0,4.86,4.89,4.84,4.94,4.94,4.93,4.83
1,29051,NaN,t,NaN,NaN,NaN,2.0,NaN,t,Private room in condo,...,85,255,NaN,4.82,4.88,4.83,4.93,4.93,4.88,4.79
2,44129,NaN,t,NaN,NaN,NaN,6.0,NaN,t,Entire rental unit,...,1,39,12233.0,4.88,4.81,4.88,4.89,4.89,4.95,4.59
3,44391,NaN,f,NaN,NaN,NaN,1.0,NaN,t,Entire rental unit,...,0,0,NaN,4.71,4.68,4.49,4.95,4.90,4.68,4.50
4,48373,NaN,f,NaN,NaN,NaN,1.0,NaN,t,Entire rental unit,...,0,0,NaN,5.00,5.00,5.00,5.00,5.00,4.60,5.00


In [106]:
def clean_percentage(value):
    """
    Convert percentage values like '95%' into numeric values like 95.0.
    """
    if pd.isna(value):
        return pd.NA
    
    value = str(value).replace("%", "").strip()
    return pd.to_numeric(value, errors="coerce")


def clean_airbnb_boolean(value):
    """
    Convert Airbnb boolean values t/f into True/False.
    """
    if pd.isna(value):
        return pd.NA
    
    value = str(value).strip().lower()
    
    if value == "t":
        return True
    if value == "f":
        return False
    
    return pd.NA

In [107]:
detailed_enrichment_clean = pd.DataFrame()

detailed_enrichment_clean["listing_id"] = detailed_enrichment["listing_id"].astype("Int64")

if "host_is_superhost" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_is_superhost"] = detailed_enrichment["host_is_superhost"].apply(clean_airbnb_boolean)

if "host_response_time" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_response_time"] = detailed_enrichment["host_response_time"].apply(standardize_text)

if "host_response_rate" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_response_rate"] = detailed_enrichment["host_response_rate"].apply(clean_percentage)

if "host_acceptance_rate" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_acceptance_rate"] = detailed_enrichment["host_acceptance_rate"].apply(clean_percentage)

if "host_listings_count" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_listings_count"] = pd.to_numeric(
        detailed_enrichment["host_listings_count"], errors="coerce"
    )

if "host_total_listings_count" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_total_listings_count"] = pd.to_numeric(
        detailed_enrichment["host_total_listings_count"], errors="coerce"
    )

if "property_type" in detailed_enrichment.columns:
    detailed_enrichment_clean["property_type"] = detailed_enrichment["property_type"].apply(standardize_text)

if "accommodates" in detailed_enrichment.columns:
    detailed_enrichment_clean["accommodates"] = pd.to_numeric(
        detailed_enrichment["accommodates"], errors="coerce"
    )

if "bathrooms_text" in detailed_enrichment.columns:
    detailed_enrichment_clean["bathrooms_text"] = detailed_enrichment["bathrooms_text"].apply(standardize_text)

if "bedrooms" in detailed_enrichment.columns:
    detailed_enrichment_clean["bedrooms"] = pd.to_numeric(
        detailed_enrichment["bedrooms"], errors="coerce"
    )

if "beds" in detailed_enrichment.columns:
    detailed_enrichment_clean["beds"] = pd.to_numeric(
        detailed_enrichment["beds"], errors="coerce"
    )

if "amenities" in detailed_enrichment.columns:
    detailed_enrichment_clean["amenities"] = detailed_enrichment["amenities"].apply(standardize_text)
    detailed_enrichment_clean["amenities_count"] = detailed_enrichment_clean["amenities"].apply(
        lambda x: len(str(x).split(",")) if pd.notna(x) else pd.NA
    )

if "instant_bookable" in detailed_enrichment.columns:
    detailed_enrichment_clean["instant_bookable"] = detailed_enrichment["instant_bookable"].apply(clean_airbnb_boolean)

review_score_columns = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for column in review_score_columns:
    if column in detailed_enrichment.columns:
        detailed_enrichment_clean[column] = pd.to_numeric(
            detailed_enrichment[column], errors="coerce"
        )

if "host_since" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_since"] = pd.to_datetime(
        detailed_enrichment["host_since"], errors="coerce"
    )

if "host_identity_verified" in detailed_enrichment.columns:
    detailed_enrichment_clean["host_identity_verified"] = detailed_enrichment["host_identity_verified"].apply(clean_airbnb_boolean)

if "bathrooms" in detailed_enrichment.columns:
    detailed_enrichment_clean["bathrooms"] = pd.to_numeric(
        detailed_enrichment["bathrooms"], errors="coerce"
    )

for column in [
    "availability_30",
    "availability_60",
    "availability_90",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d"
]:
    if column in detailed_enrichment.columns:
        detailed_enrichment_clean[column] = pd.to_numeric(
            detailed_enrichment[column], errors="coerce"
        )
        
detailed_enrichment_clean.head()

,listing_id,host_is_superhost,host_response_time,host_response_rate,host_acceptance_rate,host_listings_count,host_total_listings_count,property_type,accommodates,bathrooms_text,...,host_since,host_identity_verified,bathrooms,availability_30,availability_60,availability_90,number_of_reviews_l30d,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d
0,28871,True,<NA>,<NA>,<NA>,2.0,NaN,Private room in rental unit,2,1 shared bath,...,NaT,True,NaN,1,1,1,5,95,255,23970.0
1,29051,True,<NA>,<NA>,<NA>,2.0,NaN,Private room in condo,2,1 shared bath,...,NaT,True,NaN,0,2,2,6,85,255,NaN
2,44129,True,<NA>,<NA>,<NA>,6.0,NaN,Entire rental unit,2,1.5 baths,...,NaT,True,NaN,3,3,3,0,1,39,12233.0
3,44391,False,<NA>,<NA>,<NA>,1.0,NaN,Entire rental unit,4,1.5 baths,...,NaT,True,NaN,0,0,0,0,0,0,NaN
4,48373,False,<NA>,<NA>,<NA>,1.0,NaN,Entire rental unit,4,1.5 baths,...,NaT,True,NaN,0,0,0,0,0,0,NaN


In [108]:
detailed_enrichment_validation = {
    "row_count": len(detailed_enrichment_clean),
    "column_count": detailed_enrichment_clean.shape[1],
    "unique_listing_ids": detailed_enrichment_clean["listing_id"].nunique(),
    "duplicate_listing_ids": detailed_enrichment_clean["listing_id"].duplicated().sum(),
    "missing_listing_ids": detailed_enrichment_clean["listing_id"].isna().sum()
}

detailed_enrichment_validation_df = pd.DataFrame(
    list(detailed_enrichment_validation.items()),
    columns=["check_name", "check_value"]
)

detailed_enrichment_validation_df

,check_name,check_value
0,row_count,10369
1,column_count,32
2,unique_listing_ids,10369
3,duplicate_listing_ids,0
4,missing_listing_ids,0


### Base Listings + Detailed Enrichment Join

The cleaned summary listings table is left-joined with the cleaned detailed listings enrichment table.

A left join is used because the base table has broader listing coverage.  
This keeps all base listings even when detailed metadata is unavailable.

In [109]:
listing_master_base = summary_listings_clean.merge(
    detailed_enrichment_clean,
    on="listing_id",
    how="left",
    indicator=True
)

listing_master_base.head()

,listing_id,listing_name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,host_identity_verified,bathrooms,availability_30,availability_60,availability_90,number_of_reviews_l30d,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,_merge
0,28871,Comfortable double room,124245,1462510208428038912,Edwin,<NA>,Centrum-West,52.367750,4.89092,Private room,...,True,NaN,1.0,1.0,1.0,5.0,95.0,255.0,23970.0,both
1,29051,Comfortable single / double room,124245,1462510208428038912,Edwin,<NA>,Centrum-Oost,52.365840,4.89111,Private room,...,True,NaN,0.0,2.0,2.0,6.0,85.0,255.0,NaN,both
2,44129,Luxury design with canal view,187728,1462512125293144576,Tanya,<NA>,Centrum-West,52.382110,4.88630,Entire home/apt,...,True,NaN,3.0,3.0,3.0,0.0,1.0,39.0,12233.0,both
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,1462512336903650304,Jan,<NA>,Centrum-Oost,52.371680,4.91471,Entire home/apt,...,True,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,both
4,48373,Cozy family home in Amsterdam South,220434,1462513002935125760,Vesna & Misha,<NA>,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,...,True,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,both


In [110]:
join_summary = listing_master_base["_merge"].value_counts().reset_index()
join_summary.columns = ["join_status", "row_count"]

join_summary

,join_status,row_count
0,both,10369
1,left_only,96
2,right_only,0


In [111]:
join_summary_output_path = REPORTS_PATH / "amsterdam_detailed_enrichment_join_summary.csv"

join_summary.to_csv(join_summary_output_path, index=False)

print(f"Detailed enrichment join summary saved to: {join_summary_output_path}")

Detailed enrichment join summary saved to: ..\reports\data_quality\amsterdam_detailed_enrichment_join_summary.csv


In [112]:
listing_master_base = listing_master_base.drop(columns=["_merge"])

listing_master_base_output_path = PROCESSED_DATA_PATH / "listing_master_base.parquet"

listing_master_base.to_parquet(listing_master_base_output_path, index=False)

print(f"Listing master base saved to: {listing_master_base_output_path}")

Listing master base saved to: ..\data\processed\amsterdam\listing_master_base.parquet


In [113]:
[
    column for column in listing_master_base.columns 
    if "estimated" in column or "availability_" in column
]

['availability_365',
 'availability_30',
 'availability_60',
 'availability_90',
 'estimated_occupancy_l365d',
 'estimated_revenue_l365d']

### Detailed Enrichment Observations

The cleaned base listing table was enriched with selected detailed listing attributes related to host behavior, property characteristics, amenities, booking settings, and review scores.

A left join was used to preserve all listings from the base summary listings table.  
Most listings matched successfully with detailed metadata, while a small number of base listings did not have matching detailed records.  
These unmatched listings are retained to preserve market coverage and will be handled as partial metadata records in downstream analysis.

This approach balances coverage and richness, which is important for market-level analytics and machine learning readiness.

## 7. Calendar Data Cleaning and Aggregation

This section cleans the calendar dataset and aggregates daily listing-level availability and price records into listing-level metrics.

The calendar file is used to estimate availability patterns, occupancy proxy, average daily price, weekend pricing, and potential annual revenue.

Unavailable days are treated as an occupancy proxy, but this is a limitation because unavailable days may include both booked dates and host-blocked dates.

In [92]:
calendar = datasets["calendar"].copy()

print("Calendar shape:", calendar.shape)
print("Columns:")
print(calendar.columns.tolist())

calendar.head()

Calendar shape: (3819725, 5)
Columns:
['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']


,listing_id,date,available,minimum_nights,maximum_nights
0,28871,2026-06-16,f,2,730
1,28871,2026-06-17,f,2,730
2,28871,2026-06-18,f,2,730
3,28871,2026-06-19,f,2,730
4,28871,2026-06-20,f,2,730


In [93]:
calendar_clean = pd.DataFrame()

calendar_clean["listing_id"] = calendar["listing_id"].astype("Int64")
calendar_clean["date"] = pd.to_datetime(calendar["date"], errors="coerce")
calendar_clean["available"] = calendar["available"].apply(clean_airbnb_boolean)

calendar_clean["minimum_nights"] = pd.to_numeric(calendar["minimum_nights"], errors="coerce")
calendar_clean["maximum_nights"] = pd.to_numeric(calendar["maximum_nights"], errors="coerce")

calendar_clean["day_of_week"] = calendar_clean["date"].dt.day_name()
calendar_clean["month"] = calendar_clean["date"].dt.month
calendar_clean["month_name"] = calendar_clean["date"].dt.month_name()
calendar_clean["is_weekend"] = calendar_clean["day_of_week"].isin(["Saturday", "Sunday"])

calendar_clean.head()

,listing_id,date,available,minimum_nights,maximum_nights,day_of_week,month,month_name,is_weekend
0,28871,2026-06-16,False,2,730,Tuesday,6,June,False
1,28871,2026-06-17,False,2,730,Wednesday,6,June,False
2,28871,2026-06-18,False,2,730,Thursday,6,June,False
3,28871,2026-06-19,False,2,730,Friday,6,June,False
4,28871,2026-06-20,False,2,730,Saturday,6,June,True


In [94]:
calendar_validation = {
    "row_count": len(calendar_clean),
    "column_count": calendar_clean.shape[1],
    "unique_listing_ids": calendar_clean["listing_id"].nunique(),
    "missing_listing_ids": calendar_clean["listing_id"].isna().sum(),
    "missing_dates": calendar_clean["date"].isna().sum(),
    "missing_available": calendar_clean["available"].isna().sum(),
    "missing_minimum_nights": calendar_clean["minimum_nights"].isna().sum(),
    "missing_maximum_nights": calendar_clean["maximum_nights"].isna().sum(),
    "invalid_minimum_nights": (calendar_clean["minimum_nights"] < 0).sum(),
    "invalid_maximum_nights": (calendar_clean["maximum_nights"] < 0).sum()
}

calendar_validation_df = pd.DataFrame(
    list(calendar_validation.items()),
    columns=["check_name", "check_value"]
)

calendar_validation_df

,check_name,check_value
0,row_count,3819725
1,column_count,9
2,unique_listing_ids,10465
3,missing_listing_ids,0
4,missing_dates,0
5,missing_available,0
6,missing_minimum_nights,0
7,missing_maximum_nights,0
8,invalid_minimum_nights,0
9,invalid_maximum_nights,0


In [95]:
calendar_validation_output_path = REPORTS_PATH / "amsterdam_calendar_clean_validation.csv"

calendar_validation_df.to_csv(calendar_validation_output_path, index=False)

print(f"Calendar validation saved to: {calendar_validation_output_path}")

Calendar validation saved to: ..\reports\data_quality\amsterdam_calendar_clean_validation.csv


In [96]:
calendar_listing_metrics = (
    calendar_clean
    .groupby("listing_id")
    .agg(
        calendar_days=("date", "count"),
        available_days=("available", lambda x: (x == True).sum()),
        unavailable_days=("available", lambda x: (x == False).sum()),
        weekend_available_days=("available", lambda x: (
            (x == True) & (calendar_clean.loc[x.index, "is_weekend"] == True)
        ).sum()),
        weekend_total_days=("is_weekend", lambda x: (x == True).sum()),
        weekday_available_days=("available", lambda x: (
            (x == True) & (calendar_clean.loc[x.index, "is_weekend"] == False)
        ).sum()),
        weekday_total_days=("is_weekend", lambda x: (x == False).sum()),
        avg_minimum_nights=("minimum_nights", "mean"),
        median_minimum_nights=("minimum_nights", "median"),
        avg_maximum_nights=("maximum_nights", "mean"),
        median_maximum_nights=("maximum_nights", "median")
    )
    .reset_index()
)

calendar_listing_metrics["availability_rate"] = (
    calendar_listing_metrics["available_days"] / calendar_listing_metrics["calendar_days"]
).round(4)

calendar_listing_metrics["occupancy_proxy"] = (
    calendar_listing_metrics["unavailable_days"] / calendar_listing_metrics["calendar_days"]
).round(4)

calendar_listing_metrics["weekend_availability_rate"] = (
    calendar_listing_metrics["weekend_available_days"] / calendar_listing_metrics["weekend_total_days"]
).round(4)

calendar_listing_metrics["weekday_availability_rate"] = (
    calendar_listing_metrics["weekday_available_days"] / calendar_listing_metrics["weekday_total_days"]
).round(4)

calendar_listing_metrics.head()

,listing_id,calendar_days,available_days,unavailable_days,weekend_available_days,weekend_total_days,weekday_available_days,weekday_total_days,avg_minimum_nights,median_minimum_nights,avg_maximum_nights,median_maximum_nights,availability_rate,occupancy_proxy,weekend_availability_rate,weekday_availability_rate
0,28871,365,11,354,5,104,6,261,1.994521,2.0,730.0,730.0,0.0301,0.9699,0.0481,0.0230
1,29051,365,2,363,1,104,1,261,1.980822,2.0,730.0,730.0,0.0055,0.9945,0.0096,0.0038
2,44129,365,3,362,1,104,2,261,4.983562,5.0,1125.0,1125.0,0.0082,0.9918,0.0096,0.0077
3,44391,365,0,365,0,104,0,261,3.000000,3.0,730.0,730.0,0.0000,1.0000,0.0000,0.0000
4,48373,365,0,365,0,104,0,261,3.000000,3.0,1125.0,1125.0,0.0000,1.0000,0.0000,0.0000


In [97]:
calendar_metrics_validation = {
    "row_count": len(calendar_listing_metrics),
    "unique_listing_ids": calendar_listing_metrics["listing_id"].nunique(),
    "missing_listing_ids": calendar_listing_metrics["listing_id"].isna().sum(),
    "min_calendar_days": calendar_listing_metrics["calendar_days"].min(),
    "max_calendar_days": calendar_listing_metrics["calendar_days"].max(),
    "avg_availability_rate": round(calendar_listing_metrics["availability_rate"].mean(), 4),
    "avg_occupancy_proxy": round(calendar_listing_metrics["occupancy_proxy"].mean(), 4),
    "avg_weekend_availability_rate": round(calendar_listing_metrics["weekend_availability_rate"].mean(), 4),
    "avg_weekday_availability_rate": round(calendar_listing_metrics["weekday_availability_rate"].mean(), 4)
}

calendar_metrics_validation_df = pd.DataFrame(
    list(calendar_metrics_validation.items()),
    columns=["check_name", "check_value"]
)

calendar_metrics_validation_df

,check_name,check_value
0,row_count,10465.0000
1,unique_listing_ids,10465.0000
2,missing_listing_ids,0.0000
3,min_calendar_days,365.0000
4,max_calendar_days,365.0000
5,avg_availability_rate,0.2615
6,avg_occupancy_proxy,0.7385
7,avg_weekend_availability_rate,0.2599
8,avg_weekday_availability_rate,0.2621


In [98]:
calendar_metrics_output_path = PROCESSED_DATA_PATH / "calendar_listing_metrics.parquet"

calendar_listing_metrics.to_parquet(calendar_metrics_output_path, index=False)

print(f"Calendar listing metrics saved to: {calendar_metrics_output_path}")

Calendar listing metrics saved to: ..\data\processed\amsterdam\calendar_listing_metrics.parquet


In [99]:
calendar_metrics_validation_output_path = REPORTS_PATH / "amsterdam_calendar_metrics_validation.csv"

calendar_metrics_validation_df.to_csv(calendar_metrics_validation_output_path, index=False)

print(f"Calendar metrics validation saved to: {calendar_metrics_validation_output_path}")

Calendar metrics validation saved to: ..\reports\data_quality\amsterdam_calendar_metrics_validation.csv


## Decision 002: Calendar Pricing Limitation

**Decision:** Use the calendar dataset for availability and stay-policy analysis, but not direct daily pricing analysis.

**Reasoning:**  
The Amsterdam calendar file contains listing_id, date, available, minimum_nights, and maximum_nights. It does not contain daily price or adjusted_price columns.

**Trade-off:**  
Weekend price premium and calendar-based revenue cannot be calculated directly from the calendar data. Any revenue metric created later will use listing-level price as a proxy and will be clearly labeled as an estimate.

**Impact:**  
The project will focus calendar-derived features on availability rate, occupancy proxy, weekend/weekday availability, and minimum/maximum stay patterns.

### Calendar Data Limitation

The Amsterdam calendar dataset does not contain daily price or adjusted price columns.  
Therefore, this project does not calculate direct daily pricing trends, weekend price premiums, or calendar-based revenue from the calendar file.

The calendar dataset is used for availability analysis, occupancy proxy calculation, weekend/weekday availability comparison, and stay-policy analysis using minimum and maximum nights.

Any future revenue estimate will use listing-level price as a proxy and will be clearly labeled as an estimate.

## 8. Listing Master Table with Calendar Metrics

This section enriches the listing master base table with listing-level calendar metrics.

The calendar metrics include available days, unavailable days, availability rate, occupancy proxy, weekend availability, weekday availability, and stay-policy metrics.

A left join is used to preserve the full base listing universe.

In [114]:
listing_master_with_calendar = listing_master_base.merge(
    calendar_listing_metrics,
    on="listing_id",
    how="left",
    indicator=True
)

listing_master_with_calendar.head()

,listing_id,listing_name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,weekday_total_days,avg_minimum_nights,median_minimum_nights,avg_maximum_nights,median_maximum_nights,availability_rate,occupancy_proxy,weekend_availability_rate,weekday_availability_rate,_merge
0,28871,Comfortable double room,124245,1462510208428038912,Edwin,<NA>,Centrum-West,52.367750,4.89092,Private room,...,261,1.994521,2.0,730.0,730.0,0.0301,0.9699,0.0481,0.0230,both
1,29051,Comfortable single / double room,124245,1462510208428038912,Edwin,<NA>,Centrum-Oost,52.365840,4.89111,Private room,...,261,1.980822,2.0,730.0,730.0,0.0055,0.9945,0.0096,0.0038,both
2,44129,Luxury design with canal view,187728,1462512125293144576,Tanya,<NA>,Centrum-West,52.382110,4.88630,Entire home/apt,...,261,4.983562,5.0,1125.0,1125.0,0.0082,0.9918,0.0096,0.0077,both
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,1462512336903650304,Jan,<NA>,Centrum-Oost,52.371680,4.91471,Entire home/apt,...,261,3.000000,3.0,730.0,730.0,0.0000,1.0000,0.0000,0.0000,both
4,48373,Cozy family home in Amsterdam South,220434,1462513002935125760,Vesna & Misha,<NA>,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,...,261,3.000000,3.0,1125.0,1125.0,0.0000,1.0000,0.0000,0.0000,both


In [115]:
calendar_join_summary = listing_master_with_calendar["_merge"].value_counts().reset_index()
calendar_join_summary.columns = ["join_status", "row_count"]

calendar_join_summary

,join_status,row_count
0,both,10465
1,left_only,0
2,right_only,0


In [116]:
calendar_join_summary_output_path = REPORTS_PATH / "amsterdam_calendar_join_summary.csv"

calendar_join_summary.to_csv(calendar_join_summary_output_path, index=False)

print(f"Calendar join summary saved to: {calendar_join_summary_output_path}")

Calendar join summary saved to: ..\reports\data_quality\amsterdam_calendar_join_summary.csv


In [117]:
listing_master_with_calendar = listing_master_with_calendar.drop(columns=["_merge"])

In [119]:
# Ensure price and unavailable_days are numeric before revenue proxy calculation
listing_master_with_calendar["price_numeric"] = pd.to_numeric(
    listing_master_with_calendar["price"],
    errors="coerce"
)

listing_master_with_calendar["unavailable_days_numeric"] = pd.to_numeric(
    listing_master_with_calendar["unavailable_days"],
    errors="coerce"
)

# Flag whether listing-level price is available
listing_master_with_calendar["listing_price_available_flag"] = (
    listing_master_with_calendar["price_numeric"].notna()
)

# Revenue proxy calculation
# Important: this is not actual revenue, only an estimate using listing-level price and unavailable days
listing_master_with_calendar["estimated_revenue_proxy"] = (
    listing_master_with_calendar["unavailable_days_numeric"] *
    listing_master_with_calendar["price_numeric"]
).round(2)

# Host portfolio segmentation
listing_master_with_calendar["host_portfolio_segment"] = pd.cut(
    listing_master_with_calendar["calculated_host_listings_count"],
    bins=[0, 1, 5, 20, float("inf")],
    labels=[
        "Single-listing host",
        "Small portfolio host",
        "Medium portfolio host",
        "Large portfolio host"
    ],
    include_lowest=True
)

# Availability segmentation
listing_master_with_calendar["availability_segment"] = pd.cut(
    listing_master_with_calendar["availability_rate"],
    bins=[-0.01, 0.25, 0.50, 0.75, 1.00],
    labels=[
        "Low availability",
        "Moderate availability",
        "High availability",
        "Very high availability"
    ]
)

listing_master_with_calendar[
    [
        "listing_id",
        "price_numeric",
        "unavailable_days_numeric",
        "estimated_revenue_proxy",
        "host_portfolio_segment",
        "availability_segment"
    ]
].head()

,listing_id,price_numeric,unavailable_days_numeric,estimated_revenue_proxy,host_portfolio_segment,availability_segment
0,28871,94.0,354,33276.0,Small portfolio host,Low availability
1,29051,NaN,363,NaN,Small portfolio host,Low availability
2,44129,314.0,362,113668.0,Small portfolio host,Low availability
3,44391,NaN,365,NaN,Single-listing host,Low availability
4,48373,NaN,365,NaN,Single-listing host,Low availability


In [120]:
listing_master_calendar_validation = {
    "row_count": len(listing_master_with_calendar),
    "column_count": listing_master_with_calendar.shape[1],
    "unique_listing_ids": listing_master_with_calendar["listing_id"].nunique(),
    "duplicate_listing_ids": listing_master_with_calendar["listing_id"].duplicated().sum(),
    "missing_listing_ids": listing_master_with_calendar["listing_id"].isna().sum(),
    "missing_calendar_days": listing_master_with_calendar["calendar_days"].isna().sum(),
    "missing_availability_rate": listing_master_with_calendar["availability_rate"].isna().sum(),
    "missing_occupancy_proxy": listing_master_with_calendar["occupancy_proxy"].isna().sum(),
    "missing_estimated_revenue_proxy": listing_master_with_calendar["estimated_revenue_proxy"].isna().sum()
}

listing_master_calendar_validation_df = pd.DataFrame(
    list(listing_master_calendar_validation.items()),
    columns=["check_name", "check_value"]
)

listing_master_calendar_validation_df

,check_name,check_value
0,row_count,10465
1,column_count,74
2,unique_listing_ids,10465
3,duplicate_listing_ids,0
4,missing_listing_ids,0
5,missing_calendar_days,0
6,missing_availability_rate,0
7,missing_occupancy_proxy,0
8,missing_estimated_revenue_proxy,3994


In [121]:
listing_master_calendar_validation_output_path = REPORTS_PATH / "amsterdam_listing_master_calendar_validation.csv"

listing_master_calendar_validation_df.to_csv(listing_master_calendar_validation_output_path, index=False)

print(f"Listing master calendar validation saved to: {listing_master_calendar_validation_output_path}")

Listing master calendar validation saved to: ..\reports\data_quality\amsterdam_listing_master_calendar_validation.csv


In [122]:
listing_master_with_calendar_output_path = PROCESSED_DATA_PATH / "listing_master_with_calendar.parquet"

listing_master_with_calendar.to_parquet(listing_master_with_calendar_output_path, index=False)

print(f"Listing master with calendar saved to: {listing_master_with_calendar_output_path}")

Listing master with calendar saved to: ..\data\processed\amsterdam\listing_master_with_calendar.parquet


### Listing Master with Calendar Observations

Calendar-derived metrics were successfully joined into the listing master table.

The enriched table now includes listing-level availability, occupancy proxy, weekend/weekday availability, and stay-policy metrics.  
This improves the analytical value of the listing master table by connecting listing attributes with market availability behavior.

A revenue proxy was created using listing-level price and unavailable days.  
This metric is not treated as actual revenue because unavailable days may include host-blocked dates and the calendar file does not contain daily price fields.

## 9. Review Data Cleaning and Aggregation

This section cleans the detailed reviews dataset and aggregates review-level records into listing-level review metrics.

The detailed reviews file is used instead of the summary reviews file because it contains review identifiers, reviewer information, dates, and review comments.

These metrics will support demand-side analysis, review activity analysis, and later NLP/AI experiments.

In [123]:
detailed_reviews = datasets["detailed_reviews"].copy()

print("Detailed reviews shape:", detailed_reviews.shape)
print("Columns:")
print(detailed_reviews.columns.tolist())

detailed_reviews.head()

Detailed reviews shape: (545162, 6)
Columns:
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,28871,82539,2010-08-22,163752,Dave,Very nice place and people. Great location! ...
1,28871,21518912,2014-10-19,20597737,Zuzana,We were at the double room and I don't have an...
2,28871,26293687,2015-02-09,20543694,Melissa,"Edwin is friendly and humorous guide, his hous..."
3,28871,32450201,2015-05-18,26175845,Franz,"Nos ha gustado mucho el sitio de Edwin, esta i..."
4,28871,32686822,2015-05-20,29217313,Maria,I didn't actually see Edwin because he was out...


In [124]:
review_dates_check = pd.to_datetime(detailed_reviews["date"], errors="coerce")

print("Earliest review date:", review_dates_check.min())
print("Latest review date:", review_dates_check.max())
print("Missing review dates:", review_dates_check.isna().sum())

Earliest review date: 2010-08-16 00:00:00
Latest review date: 2026-06-28 00:00:00
Missing review dates: 0


In [125]:
reviews_clean = pd.DataFrame()

reviews_clean["listing_id"] = detailed_reviews["listing_id"].astype("Int64")
reviews_clean["review_id"] = detailed_reviews["id"].astype("Int64")
reviews_clean["review_date"] = pd.to_datetime(detailed_reviews["date"], errors="coerce")
reviews_clean["reviewer_id"] = detailed_reviews["reviewer_id"].astype("Int64")
reviews_clean["reviewer_name"] = detailed_reviews["reviewer_name"].apply(standardize_text)

reviews_clean["comments"] = detailed_reviews["comments"].apply(standardize_text)

reviews_clean["has_comment"] = reviews_clean["comments"].notna()
reviews_clean["comment_length"] = reviews_clean["comments"].apply(
    lambda x: len(str(x)) if pd.notna(x) else 0
)

reviews_clean.head()

,listing_id,review_id,review_date,reviewer_id,reviewer_name,comments,has_comment,comment_length
0,28871,82539,2010-08-22,163752,Dave,Very nice place and people. Great location! ...,True,73
1,28871,21518912,2014-10-19,20597737,Zuzana,We were at the double room and I don't have an...,True,259
2,28871,26293687,2015-02-09,20543694,Melissa,"Edwin is friendly and humorous guide, his hous...",True,166
3,28871,32450201,2015-05-18,26175845,Franz,"Nos ha gustado mucho el sitio de Edwin, esta i...",True,276
4,28871,32686822,2015-05-20,29217313,Maria,I didn't actually see Edwin because he was out...,True,436


In [126]:
reviews_clean_validation = {
    "row_count": len(reviews_clean),
    "column_count": reviews_clean.shape[1],
    "unique_review_ids": reviews_clean["review_id"].nunique(),
    "duplicate_review_ids": reviews_clean["review_id"].duplicated().sum(),
    "missing_review_ids": reviews_clean["review_id"].isna().sum(),
    "missing_listing_ids": reviews_clean["listing_id"].isna().sum(),
    "missing_review_dates": reviews_clean["review_date"].isna().sum(),
    "missing_comments": reviews_clean["comments"].isna().sum(),
    "reviews_with_comments": reviews_clean["has_comment"].sum()
}

reviews_clean_validation_df = pd.DataFrame(
    list(reviews_clean_validation.items()),
    columns=["check_name", "check_value"]
)

reviews_clean_validation_df

,check_name,check_value
0,row_count,545162
1,column_count,8
2,unique_review_ids,545162
3,duplicate_review_ids,0
4,missing_review_ids,0
5,missing_listing_ids,0
6,missing_review_dates,0
7,missing_comments,37
8,reviews_with_comments,545125


In [127]:
reviews_clean_validation_output_path = REPORTS_PATH / "amsterdam_reviews_clean_validation.csv"

reviews_clean_validation_df.to_csv(reviews_clean_validation_output_path, index=False)

print(f"Reviews clean validation saved to: {reviews_clean_validation_output_path}")

Reviews clean validation saved to: ..\reports\data_quality\amsterdam_reviews_clean_validation.csv


In [128]:
reviews_clean_output_path = PROCESSED_DATA_PATH / "reviews_clean.parquet"

reviews_clean.to_parquet(reviews_clean_output_path, index=False)

print(f"Cleaned reviews saved to: {reviews_clean_output_path}")

Cleaned reviews saved to: ..\data\processed\amsterdam\reviews_clean.parquet


### Review Metrics Aggregation

The cleaned reviews table is aggregated to listing level.  
This creates demand-side features that can be joined into the listing master table.

In [130]:
SNAPSHOT_DATE = pd.to_datetime("2026-06-15")
recent_review_start_date = SNAPSHOT_DATE - pd.Timedelta(days=365)

review_listing_metrics = (
    reviews_clean
    .groupby("listing_id")
    .agg(
        detailed_review_count=("review_id", "count"),
        unique_reviewer_count=("reviewer_id", "nunique"),
        first_detailed_review_date=("review_date", "min"),
        latest_detailed_review_date=("review_date", "max"),
        reviews_with_comments=("has_comment", "sum"),
        avg_comment_length=("comment_length", "mean"),
        median_comment_length=("comment_length", "median")
    )
    .reset_index()
)

recent_reviews = (
    reviews_clean[
        (reviews_clean["review_date"] >= recent_review_start_date) &
        (reviews_clean["review_date"] <= SNAPSHOT_DATE)
    ]
    .groupby("listing_id")
    .agg(
        detailed_reviews_last_365d=("review_id", "count")
    )
    .reset_index()
)

review_listing_metrics = review_listing_metrics.merge(
    recent_reviews,
    on="listing_id",
    how="left"
)

review_listing_metrics["detailed_reviews_last_365d"] = (
    review_listing_metrics["detailed_reviews_last_365d"]
    .fillna(0)
    .astype(int)
)

review_listing_metrics["comment_coverage_rate"] = (
    review_listing_metrics["reviews_with_comments"] / 
    review_listing_metrics["detailed_review_count"]
).round(4)

review_listing_metrics["review_history_days"] = (
    SNAPSHOT_DATE - review_listing_metrics["first_detailed_review_date"]
).dt.days

review_listing_metrics["review_history_years"] = (
    review_listing_metrics["review_history_days"] / 365.25
).round(2)

# Safe numeric calculation for average reviews per year
review_history_years_safe = (
    pd.to_numeric(review_listing_metrics["review_history_years"], errors="coerce")
    .replace(0, float("nan"))
)

review_listing_metrics["avg_reviews_per_year"] = (
    pd.to_numeric(review_listing_metrics["detailed_review_count"], errors="coerce") /
    review_history_years_safe
).round(2)

review_listing_metrics.head()

,listing_id,detailed_review_count,unique_reviewer_count,first_detailed_review_date,latest_detailed_review_date,reviews_with_comments,avg_comment_length,median_comment_length,detailed_reviews_last_365d,comment_coverage_rate,review_history_days,review_history_years,avg_reviews_per_year
0,28871,799,796,2010-08-22,2026-06-01,799,202.556946,163.0,89,1.0,5776,15.81,50.54
1,29051,906,894,2011-03-16,2026-06-01,906,241.263797,183.5,82,1.0,5570,15.25,59.41
2,44129,186,182,2010-08-16,2026-04-30,186,376.424731,296.5,5,1.0,5782,15.83,11.75
3,44391,42,42,2010-09-16,2022-08-20,42,242.071429,199.0,0,1.0,5751,15.75,2.67
4,48373,5,5,2023-07-21,2024-04-28,5,272.200000,60.0,0,1.0,1060,2.90,1.72


In [131]:
review_metrics_validation = {
    "row_count": len(review_listing_metrics),
    "unique_listing_ids": review_listing_metrics["listing_id"].nunique(),
    "missing_listing_ids": review_listing_metrics["listing_id"].isna().sum(),
    "min_detailed_review_count": review_listing_metrics["detailed_review_count"].min(),
    "max_detailed_review_count": review_listing_metrics["detailed_review_count"].max(),
    "avg_detailed_review_count": round(review_listing_metrics["detailed_review_count"].mean(), 2),
    "total_reviews_last_365d": review_listing_metrics["detailed_reviews_last_365d"].sum(),
    "avg_comment_coverage_rate": round(review_listing_metrics["comment_coverage_rate"].mean(), 4),
    "missing_avg_reviews_per_year": review_listing_metrics["avg_reviews_per_year"].isna().sum()
}

review_metrics_validation_df = pd.DataFrame(
    list(review_metrics_validation.items()),
    columns=["check_name", "check_value"]
)

review_metrics_validation_df

,check_name,check_value
0,row_count,9432.0000
1,unique_listing_ids,9432.0000
2,missing_listing_ids,0.0000
3,min_detailed_review_count,1.0000
4,max_detailed_review_count,5603.0000
5,avg_detailed_review_count,57.8000
6,total_reviews_last_365d,94455.0000
7,avg_comment_coverage_rate,0.9999
8,missing_avg_reviews_per_year,3.0000


In [132]:
review_metrics_output_path = PROCESSED_DATA_PATH / "review_listing_metrics.parquet"

review_listing_metrics.to_parquet(review_metrics_output_path, index=False)

print(f"Review listing metrics saved to: {review_metrics_output_path}")

Review listing metrics saved to: ..\data\processed\amsterdam\review_listing_metrics.parquet


In [133]:
review_metrics_validation_output_path = REPORTS_PATH / "amsterdam_review_metrics_validation.csv"

review_metrics_validation_df.to_csv(review_metrics_validation_output_path, index=False)

print(f"Review metrics validation saved to: {review_metrics_validation_output_path}")

Review metrics validation saved to: ..\reports\data_quality\amsterdam_review_metrics_validation.csv


### Listing Master with Review Metrics

Review metrics are joined into the listing master table using listing_id.

A left join is used to preserve all listings in the master table.  
Listings without reviews are retained and their review metrics are filled with appropriate zero or null values.

In [134]:
listing_master_with_reviews = listing_master_with_calendar.merge(
    review_listing_metrics,
    on="listing_id",
    how="left",
    indicator=True
)

review_join_summary = listing_master_with_reviews["_merge"].value_counts().reset_index()
review_join_summary.columns = ["join_status", "row_count"]

review_join_summary

,join_status,row_count
0,both,9432
1,left_only,1033
2,right_only,0


In [135]:
review_count_columns = [
    "detailed_review_count",
    "unique_reviewer_count",
    "reviews_with_comments",
    "detailed_reviews_last_365d"
]

for column in review_count_columns:
    if column in listing_master_with_reviews.columns:
        listing_master_with_reviews[column] = (
            listing_master_with_reviews[column]
            .fillna(0)
            .astype(int)
        )

listing_master_with_reviews["has_reviews"] = (
    listing_master_with_reviews["detailed_review_count"] > 0
)

listing_master_with_reviews = listing_master_with_reviews.drop(columns=["_merge"])

listing_master_with_reviews.head()

,listing_id,listing_name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,latest_detailed_review_date,reviews_with_comments,avg_comment_length,median_comment_length,detailed_reviews_last_365d,comment_coverage_rate,review_history_days,review_history_years,avg_reviews_per_year,has_reviews
0,28871,Comfortable double room,124245,1462510208428038912,Edwin,<NA>,Centrum-West,52.367750,4.89092,Private room,...,2026-06-01,799,202.556946,163.0,89,1.0,5776.0,15.81,50.54,True
1,29051,Comfortable single / double room,124245,1462510208428038912,Edwin,<NA>,Centrum-Oost,52.365840,4.89111,Private room,...,2026-06-01,906,241.263797,183.5,82,1.0,5570.0,15.25,59.41,True
2,44129,Luxury design with canal view,187728,1462512125293144576,Tanya,<NA>,Centrum-West,52.382110,4.88630,Entire home/apt,...,2026-04-30,186,376.424731,296.5,5,1.0,5782.0,15.83,11.75,True
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,1462512336903650304,Jan,<NA>,Centrum-Oost,52.371680,4.91471,Entire home/apt,...,2022-08-20,42,242.071429,199.0,0,1.0,5751.0,15.75,2.67,True
4,48373,Cozy family home in Amsterdam South,220434,1462513002935125760,Vesna & Misha,<NA>,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,...,2024-04-28,5,272.200000,60.0,0,1.0,1060.0,2.90,1.72,True


In [136]:
listing_master_reviews_validation = {
    "row_count": len(listing_master_with_reviews),
    "column_count": listing_master_with_reviews.shape[1],
    "unique_listing_ids": listing_master_with_reviews["listing_id"].nunique(),
    "duplicate_listing_ids": listing_master_with_reviews["listing_id"].duplicated().sum(),
    "missing_listing_ids": listing_master_with_reviews["listing_id"].isna().sum(),
    "listings_with_reviews": listing_master_with_reviews["has_reviews"].sum(),
    "listings_without_reviews": (~listing_master_with_reviews["has_reviews"]).sum(),
    "missing_detailed_review_count": listing_master_with_reviews["detailed_review_count"].isna().sum()
}

listing_master_reviews_validation_df = pd.DataFrame(
    list(listing_master_reviews_validation.items()),
    columns=["check_name", "check_value"]
)

listing_master_reviews_validation_df

,check_name,check_value
0,row_count,10465
1,column_count,87
2,unique_listing_ids,10465
3,duplicate_listing_ids,0
4,missing_listing_ids,0
5,listings_with_reviews,9432
6,listings_without_reviews,1033
7,missing_detailed_review_count,0


In [137]:
listing_master_reviews_validation_output_path = REPORTS_PATH / "amsterdam_listing_master_reviews_validation.csv"

listing_master_reviews_validation_df.to_csv(listing_master_reviews_validation_output_path, index=False)

print(f"Listing master reviews validation saved to: {listing_master_reviews_validation_output_path}")

Listing master reviews validation saved to: ..\reports\data_quality\amsterdam_listing_master_reviews_validation.csv


In [138]:
listing_master_with_reviews_output_path = PROCESSED_DATA_PATH / "listing_master_with_reviews.parquet"

listing_master_with_reviews.to_parquet(listing_master_with_reviews_output_path, index=False)

print(f"Listing master with reviews saved to: {listing_master_with_reviews_output_path}")

Listing master with reviews saved to: ..\data\processed\amsterdam\listing_master_with_reviews.parquet


### Review Metrics Observations

Detailed reviews were cleaned and aggregated into listing-level review metrics.  
The review metrics capture total review volume, unique reviewers, first and latest review dates, recent review activity, comment availability, and average comment length.

These features help represent demand-side activity and customer engagement.  
Listings without reviews were retained in the listing master table and flagged using the has_reviews field.